# Bengaluru Commuting & Road Traffic — Phase 1
**Purpose:** Build an evidence-led baseline of commuting and road-vehicle patterns across Bengaluru, with weekday/weekend comparisons where data supports them.

**Scope**
- Geography: Bengaluru city / urban area, with dataset-specific boundaries documented.
- Population groups: office/work commuters, school students, college students.
- Road modes: two-wheelers, cars/jeeps/vans, taxis/auto-rickshaws, school/institutional buses, other road vehicles where available.
- Public transport: retained in a separate context table; excluded from the primary *private/road-commuter* totals when the dataset supports mode separation.
- Time: all historical periods available from the source, preserving original dates and frequency.

## Critical evidence rule
Do **not** treat vehicle registrations, annual passenger totals, census mode shares, traffic violations, road crashes, modeled traffic data, and observed hourly traffic counts as interchangeable. Each answers a different question. Do not infer Monday-to-Sunday patterns from annual or cross-sectional data. The notebook labels each dataset by evidence type and only runs day-of-week analysis on records with genuine observation dates/times.

**Expected reality:** publicly accessible sources are fragmented. Census commuting data can help describe worker mode choice but is old (2011) and is not a day-by-day count. Public transport ridership and registered-vehicle series are useful context, not direct counts of all commuters on roads. The notebook records these limitations rather than manufacturing missing values.

## 1. Research questions and measurable outputs

1. What is known about work and education commuting in Bengaluru?
2. What vehicle types are represented in available traffic counts, registration data, and traffic surveys?
3. Which weekdays/weekends have higher observed traffic, and what are the mean/median counts by day?
4. How do public-transport usage indicators change over time? (Context only; excluded from the main road-vehicle totals.)
5. Which conclusions are directly observed, census-derived, or estimates?

**Unit distinctions**
- `people/trips`: people or passenger trips; these are not vehicle counts.
- `vehicles`: vehicle registrations, vehicles passing a screenline, or vehicle-hours depending on source.
- `ridership`: boardings/entries, not necessarily unique people.
- `mode share`: proportion of trips or people by mode, with the source's definition retained.

In [1]:
# Install if needed in your notebook environment:
# %pip install pandas numpy matplotlib seaborn requests openpyxl pyarrow

from pathlib import Path
from datetime import datetime, timezone
import json, re, time, hashlib, warnings
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

ROOT = Path("bengaluru_commuting_phase1")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
for p in [RAW, PROCESSED, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

RUN_DATE = pd.Timestamp.now(tz="Asia/Kolkata").strftime("%Y-%m-%d")
print("Run date:", RUN_DATE)
print("Raw data directory:", RAW.resolve())

Run date: 2026-10-10
Raw data directory: /home/dhirajpatra/Desktop/python/jupyter_notebooks/data_analysis/bengaluru_commuting_phase1/data/raw


## 2. Source register and evidence grading

The sources below were identified during initial research. OpenCity is a non-government open-data portal that republishes datasets from public agencies and other sources; validate the underlying publisher, metadata, license, date range, and geographic coverage for each resource before treating it as official.

| Source / dataset | Intended use | Evidence type | Day-of-week analysis? |
|---|---|---|---|
| Census of India 2011, Table B-28 | Worker commute mode and distance to work; mode categories include two-wheelers, cars/vans, auto/taxi, bus, train, walking, cycling | Official census counts; historical snapshot | No |
| OpenCity Bengaluru mobility dataset catalogue | Discover city mobility resources and their downloadable files | Catalogue / metadata | Only if resource has actual observation dates |
| Bengaluru RTO-wise vehicle registration data | Registered fleet trend and vehicle category context | Administrative stock/registration data | No, unless records contain dated observations at daily frequency |
| Total vehicles registered in Bengaluru | Fleet context | Administrative data | Usually no |
| BMRCL station-wise ridership (reported August 2025) | Public-transport context, kept separate | Station entries/ridership; check definitions | Only if actual date/time observations are present |
| BMTC operations/performance | Public-transport context (fleet/utilisation) | Annual or periodic operational indicators | Only if daily observations are available |
| Bengaluru traffic / mobility GitHub projects | Discovery and exploratory validation only | Non-government; source lineage must be assessed | Only when raw timestamps and measurement method are documented |
| Bengaluru Comprehensive Mobility Plan / traffic survey reports | Vehicle classification, occupancy, trip-purpose and modal-share context | Survey/model report; year and study-area boundary matter | Only if survey data has date/time fields |

Official Census metadata:
https://censusindia.gov.in/nada/index.php/catalog/13983

OpenCity Bengaluru mobility catalogue:
https://data.opencity.in/dataset/?groups=bengaluru&tags=Mobility

OpenCity catalogue filtered to CSV resources:
https://data.opencity.in/dataset/?city=&groups=bengaluru&res_format=CSV&tags=Mobility

Non-government Bengaluru traffic-data project (validate source lineage before analytical use):
https://github.com/traffic-kowshik/traffic-data-bengaluru

Example of a community traffic dataset with stated hourly records (not automatically ground-truth; inspect raw file and provenance):
https://github.com/aswinmangatt/Bengaluru-Traffic-Analysis

**Evidence grades**
- `A`: official primary data with documented methodology and clear geography/time.
- `B`: reputable intermediary with an identifiable underlying government/agency source.
- `C`: community/third-party data with some documentation but uncertain measurement lineage.
- `D`: modeled, synthetic, hand-curated, or insufficiently documented; exclude from factual citywide totals.

In [2]:
source_register = pd.DataFrame([
    {
        "source_id":"CENSUS_B28_2011",
        "dataset":"Census 2011 B-28: workers by distance and mode of travel",
        "publisher":"Office of the Registrar General & Census Commissioner, India",
        "url":"https://censusindia.gov.in/nada/index.php/catalog/13983",
        "evidence_type":"official census count / cross-sectional",
        "grade":"A",
        "use":"historical worker commute mode context",
        "public_transport_separate":True,
        "supports_day_of_week":False,
        "known_limitations":"2011 snapshot; workers, not school/college students; not daily traffic counts"
    },
    {
        "source_id":"OPENCITY_MOBILITY",
        "dataset":"OpenCity Bengaluru mobility catalogue",
        "publisher":"OpenCity; underlying publisher varies by resource",
        "url":"https://data.opencity.in/dataset/?groups=bengaluru&tags=Mobility",
        "evidence_type":"catalogue; individual resource varies",
        "grade":"B pending resource-level verification",
        "use":"discover downloadable traffic, vehicle registration, mobility and transit datasets",
        "public_transport_separate":True,
        "supports_day_of_week":"resource-dependent",
        "known_limitations":"verify original publisher, definitions, coverage, licence, date range"
    },
    {
        "source_id":"GITHUB_TRAFFIC_KOWSHIK",
        "dataset":"Traffic Data Bengaluru repository",
        "publisher":"Community GitHub project",
        "url":"https://github.com/traffic-kowshik/traffic-data-bengaluru",
        "evidence_type":"third-party repository / resource-dependent",
        "grade":"C pending provenance",
        "use":"route/transport data discovery; exploratory only",
        "public_transport_separate":True,
        "supports_day_of_week":"only if raw observations have real timestamps",
        "known_limitations":"repository is not itself proof of measured traffic counts"
    },
    {
        "source_id":"GITHUB_TRAFFIC_ANALYSIS",
        "dataset":"Bengaluru Traffic Analysis example project",
        "publisher":"Community GitHub project",
        "url":"https://github.com/aswinmangatt/Bengaluru-Traffic-Analysis",
        "evidence_type":"third-party traffic dataset / source lineage to validate",
        "grade":"C/D until independently validated",
        "use":"candidate dataset for schema/provenance review, not default citywide truth",
        "public_transport_separate":True,
        "supports_day_of_week":"claimed hourly schema; validate raw data and collection method",
        "known_limitations":"repository descriptions alone do not establish that records are real measured counts"
    }
])
source_register

,source_id,dataset,publisher,url,evidence_type,grade,use,public_transport_separate,supports_day_of_week,known_limitations
0,CENSUS_B28_2011,Census 2011 B-28: workers by distance and mode of travel,"Office of the Registrar General & Census Commissioner, India",https://censusindia.gov.in/nada/index.php/catalog/13983,official census count / cross-sectional,A,historical worker commute mode context,True,False,"2011 snapshot; workers, not school/college students; not daily traffic counts"
1,OPENCITY_MOBILITY,OpenCity Bengaluru mobility catalogue,OpenCity; underlying publisher varies by resource,https://data.opencity.in/dataset/?groups=bengaluru&tags=Mobility,catalogue; individual resource varies,B pending resource-level verification,"discover downloadable traffic, vehicle registration, mobility and transit datasets",True,resource-dependent,"verify original publisher, definitions, coverage, licence, date range"
2,GITHUB_TRAFFIC_KOWSHIK,Traffic Data Bengaluru repository,Community GitHub project,https://github.com/traffic-kowshik/traffic-data-bengaluru,third-party repository / resource-dependent,C pending provenance,route/transport data discovery; exploratory only,True,only if raw observations have real timestamps,repository is not itself proof of measured traffic counts
3,GITHUB_TRAFFIC_ANALYSIS,Bengaluru Traffic Analysis example project,Community GitHub project,https://github.com/aswinmangatt/Bengaluru-Traffic-Analysis,third-party traffic dataset / source lineage to validate,C/D until independently validated,"candidate dataset for schema/provenance review, not default citywide truth",True,claimed hourly schema; validate raw data and collection method,repository descriptions alone do not establish that records are real measured counts


## 3. Discover OpenCity datasets through its CKAN API

This cell queries the public CKAN catalogue and saves dataset metadata. It does not assume that every listed resource is a direct government source or that a downloadable CSV is suitable for analysis.

If the API is temporarily unavailable, use the catalogue URLs in the source register and save resources manually into `data/raw/`. Keep original filenames and record the download URL.

In [3]:
CKAN_API = "https://data.opencity.in/api/3/action/package_search"
queries = [
    "Bengaluru mobility",
    "Bengaluru vehicle registrations",
    "Bengaluru traffic",
    "BMRCL station-wise ridership",
    "BMTC operation performance",
    "Bengaluru mobility indicators",
    "Bengaluru RTO-wise vehicle registration",
    "Total vehicles registered Bengaluru",
    "Bengaluru traffic violations",
    "Bengaluru road crashes",
]

catalogue_rows = []
session = requests.Session()
session.headers.update({"User-Agent":"BengaluruCommutingResearch/1.0 (research notebook)"})

for q in queries:
    try:
        resp = session.get(CKAN_API, params={"q": q, "rows": 100}, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        if not payload.get("success"):
            print("Catalogue query failed:", q)
            continue
        for pkg in payload["result"].get("results", []):
            for resource in pkg.get("resources", []):
                catalogue_rows.append({
                    "query": q,
                    "package_name": pkg.get("name"),
                    "package_title": pkg.get("title"),
                    "organization": (pkg.get("organization") or {}).get("title"),
                    "package_url": "https://data.opencity.in/dataset/" + str(pkg.get("name","")),
                    "resource_name": resource.get("name"),
                    "format": resource.get("format"),
                    "resource_url": resource.get("url"),
                    "resource_id": resource.get("id"),
                    "created": resource.get("created"),
                    "last_modified": resource.get("last_modified"),
                    "description": resource.get("description"),
                })
    except Exception as e:
        print(f"Could not query {q!r}: {type(e).__name__}: {e}")

catalogue = pd.DataFrame(catalogue_rows).drop_duplicates(
    subset=["package_name", "resource_id"]
) if catalogue_rows else pd.DataFrame()

catalogue.to_csv(PROCESSED / "opencity_catalogue_discovery.csv", index=False)
print("Resources discovered:", len(catalogue))
display(catalogue[["package_title","organization","resource_name","format","resource_url","last_modified"]].head(50)
        if not catalogue.empty else "No API results. Use the catalogue links manually.")

Resources discovered: 1598


,package_title,organization,resource_name,format,resource_url,last_modified
0,Bengaluru Mobility Indicators,Directorate of Urban Land Transport (DULT),Bengaluru Zone-wise Mobility Indicators - 2011,CSV,https://data.opencity.in/dataset/b2ea98f5-e88a-4668-a144-d42ba29501c4/resource/9e224133-7275-412c-b8e8-c66436dd9d9b/download/a2b10c01-ab...,2025-11-25T12:10:55.733246
1,Bengaluru Comprehensive Mobility Plan,Directorate of Urban Land Transport (DULT),Bengaluru Comprehensive Mobility Plan (CMP) Draft - October 2019,PDF,https://data.opencity.in/dataset/1e5b4cad-bb95-4d95-b878-cbb6d9013feb/resource/0f3c3f41-ed17-4c16-94c4-e2269918586c/download/4b639b61-9e...,2025-11-25T12:00:58.669566
2,Bengaluru Comprehensive Mobility Plan,Directorate of Urban Land Transport (DULT),Bengaluru Comprehensive Mobility Plan 2020,PDF,https://data.opencity.in/dataset/1e5b4cad-bb95-4d95-b878-cbb6d9013feb/resource/10812885-6dbd-446b-8359-0348ed9be390/download/538f7e60-45...,2025-11-25T12:00:59.386608
3,Bengaluru City - Mobility Policies,Directorate of Urban Land Transport (DULT),Bengaluru Parking Policy 2.0,PDF,https://data.opencity.in/dataset/f905b8fa-1ee5-4b7d-929b-c9a41f18c2f0/resource/89fee008-d97c-455e-86f3-e7b8aefafd57/download/439abae9-d2...,2025-11-25T12:11:08.263657
4,Bengaluru City - Mobility Policies,Directorate of Urban Land Transport (DULT),"Bengaluru Comprehensive Mobility Plan, 2020",PDF,https://data.opencity.in/dataset/f905b8fa-1ee5-4b7d-929b-c9a41f18c2f0/resource/6c25f042-62e3-400a-bab6-f3daa9728991/download/58f2bd6c-2b...,2025-11-25T12:11:09.150237
5,Bengaluru Metro Stations,Bengaluru Metro Rail Corporation Ltd (BMRCL),Bengaluru Metro Stations Map - April 2023,KML,https://data.opencity.in/dataset/746940cd-cca7-45f5-a3bc-8b0034663605/resource/86e5641c-496c-43fa-8143-bae5f255afc4/download/e13b04fa-bc...,2025-11-25T12:09:53.367220
6,Bengaluru - Parking Lots,Bruhat Bengaluru Mahanagara Palike (BBMP),Bengaluru Open Parking Spots,KML,https://data.opencity.in/dataset/d7808e1d-9a51-4992-8769-f48348d94197/resource/e10421c2-7bfc-4766-b794-55cea632cb4e/download/4681133c-2c...,2025-11-25T12:13:48.856021
7,Bengaluru - Parking Lots,Bruhat Bengaluru Mahanagara Palike (BBMP),Bengaluru Parking Locations Polygons,KML,https://data.opencity.in/dataset/d7808e1d-9a51-4992-8769-f48348d94197/resource/6c4a2a77-60d1-4b71-96b1-1ded920c58d3/download/476c6c70-e4...,2025-11-25T12:13:49.318705
8,Bengaluru - Parking Lots,Bruhat Bengaluru Mahanagara Palike (BBMP),Bengaluru Parking Locations,KML,https://data.opencity.in/dataset/d7808e1d-9a51-4992-8769-f48348d94197/resource/fdfe7ba5-1f45-4fff-9fd8-39056a2d46fc/download/2e94a607-fd...,2025-11-25T12:13:49.899006
9,Bengaluru Metro Documents,Bengaluru Metro Rail Corporation Ltd (BMRCL),BMRCL Namma Metro Detailed Project Report 2016,PDF,https://data.opencity.in/dataset/ecd6fd9b-10cb-4aa0-b1b9-c85dd7f7fd79/resource/773d73b8-5642-47bb-8eaf-675b89aee7cd/download/0f15af51-bc...,2025-11-25T12:08:10.383658


## 4. Select and download candidate resources

Download only common tabular formats and preserve the raw file. A successful HTTP response does not guarantee the file is clean or trustworthy; each downloaded file must be inspected. The default below is intentionally conservative and does not recursively download every resource in the catalogue.

Edit `RESOURCE_ALLOWLIST` after reviewing the catalogue. Put a resource ID in the list only after confirming its title, publisher, licence and intended use.

In [4]:
# Start empty to avoid downloading unreviewed resources.
# Example after reviewing catalogue:
# RESOURCE_ALLOWLIST = ["resource-id-1", "resource-id-2"]
RESOURCE_ALLOWLIST = []

def safe_filename(value, max_len=100):
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("._")
    return (value or "resource")[:max_len]

download_log = []

if not catalogue.empty and RESOURCE_ALLOWLIST:
    chosen = catalogue[catalogue["resource_id"].isin(RESOURCE_ALLOWLIST)]
    for _, row in chosen.iterrows():
        url = row.get("resource_url")
        if not isinstance(url, str) or not url.startswith(("https://","http://")):
            continue
        ext = str(row.get("format","")).lower().strip()
        if ext not in {"csv","xlsx","xls","json","geojson","zip","pdf"}:
            print("Skipped unsupported/unknown format:", row.get("resource_name"), ext)
            continue
        fname = safe_filename(row.get("package_title")) + "__" + safe_filename(row.get("resource_name"))
        suffix = "." + ext
        path = RAW / (fname if fname.lower().endswith(suffix) else fname + suffix)
        try:
            r = session.get(url, timeout=90)
            r.raise_for_status()
            path.write_bytes(r.content)
            download_log.append({
                "resource_id":row.get("resource_id"), "url":url,
                "path":str(path), "bytes":len(r.content),
                "sha256":hashlib.sha256(r.content).hexdigest(),
                "downloaded_at_utc":datetime.now(timezone.utc).isoformat(),
                "http_status":r.status_code
            })
            print("Saved", path, f"({len(r.content):,} bytes)")
        except Exception as e:
            print("Download failed:", url, type(e).__name__, e)
else:
    print("No resources downloaded automatically. Review catalogue and populate RESOURCE_ALLOWLIST.")

pd.DataFrame(download_log).to_csv(PROCESSED / "download_log.csv", index=False)

No resources downloaded automatically. Review catalogue and populate RESOURCE_ALLOWLIST.


## 5. Census commuting data — metadata and safe handling

Census Table B-28 records workers by mode of travel and distance to work. Its modes include walking, bicycle, moped/scooter/motorcycle, car/jeep/van, auto-rickshaw/taxi, bus and train. It is useful for historic worker mode shares, but does not cover school/college students as a separate population and has no weekday/date-of-travel observations.

The Census download page exposes an XLSX resource. Because download endpoints can change, use the official catalogue page and download the Excel file manually if the next cell cannot locate a working resource. Do not silently substitute a third-party file.

In [19]:
CENSUS_PAGE = "https://censusindia.gov.in/nada/index.php/catalog/13983"
CENSUS_FILE = RAW / "census_2011_B28_karnataka_workers_mode_distance.xlsx"

# First try to locate a spreadsheet link on the official metadata page.
try:
    from bs4 import BeautifulSoup
    page = session.get(CENSUS_PAGE, timeout=30)
    page.raise_for_status()
    soup = BeautifulSoup(page.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        label = (a.get_text(" ", strip=True) + " " + href).lower()
        if any(x in label for x in [".xlsx", "DDW-2900B-28", "download"]):
            full = requests.compat.urljoin(CENSUS_PAGE, href)
            links.append((a.get_text(" ", strip=True), full))
    print("Candidate official Census download links:")
    for item in links[:20]:
        print(item)
except Exception as e:
    print("Could not auto-discover Census download link:", type(e).__name__, e)

if CENSUS_FILE.exists():
    print("Census workbook already present:", CENSUS_FILE)
else:
    print("If needed, download the B-28 workbook from the official page above and save it as:")
    print(CENSUS_FILE.resolve())

Could not auto-discover Census download link: SSLError HTTPSConnectionPool(host='censusindia.gov.in', port=443): Max retries exceeded with url: /nada/index.php/catalog/13983 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
Census workbook already present: bengaluru_commuting_phase1/data/raw/census_2011_B28_karnataka_workers_mode_distance.xlsx


In [21]:
# Inspect the Census workbook if you downloaded it.
if CENSUS_FILE.exists():
    xls = pd.ExcelFile(CENSUS_FILE)
    print("Sheets:", xls.sheet_names)
    for sheet in xls.sheet_names:
        preview = pd.read_excel(CENSUS_FILE, sheet_name=sheet, header=None, nrows=12)
        print("\\nSHEET:", sheet)
        display(preview)
else:
    print("Skipped: official Census workbook not present locally.")

Sheets: ['B-28']
\nSHEET: B-28


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35
0,NaN,NaN,NaN,NaN,NaN,B-28 'OTHER WORKERS' BY DISTANCE FROM RESIDENCE TO PLACE OF WORK AND MODE OF TRAVEL TO PLACE OF WORK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Table,State,Distt.,Total/,Area Name,Mode of travel,Total,NaN,NaN,Distance to place of work in kilometres,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Name,Code,Code,Rural/,NaN,NaN,NaN,NaN,NaN,No travel,NaN,NaN,0-1,NaN,NaN,2-5,NaN,NaN,6-10,NaN,NaN,11-20,NaN,NaN,21-30,NaN,NaN,31-50,NaN,NaN,51+,NaN,NaN,Distance not stated,NaN,NaN
3,NaN,NaN,NaN,Urban/,NaN,NaN,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female
4,NaN,NaN,NaN,NaN,NaN,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,B0128,29,000,Total,KARNATAKA,All Modes,13222758,9794146,3428612,3865970,2518738,1347232,2080088,1461582,618506,2835872,2153578,682294,2036041,1609562,426479,1093090,914562,178528,497796,428421,69375,351308,304056,47252,337201,296020,41181,125392,107627,17765
7,B0128,29,000,Total,KARNATAKA,On foot,3145484,2068687,1076797,0,0,0,1550028,1008955,541073,1054869,684502,370367,540587,375230,165357,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,B0128,29,000,Total,KARNATAKA,Bicycle,748108,702745,45363,0,0,0,99227,92276,6951,360665,341230,19435,160661,151837,8824,64734,60152,4582,62821,57250,5571,0,0,0,0,0,0,0,0,0
9,B0128,29,000,Total,KARNATAKA,Moped/Scooter/Motor Cycle,1701363,1561606,139757,0,0,0,224185,206012,18173,618608,560571,58037,402054,368663,33391,258237,240114,18123,78503,74350,4153,53476,50394,3082,22555,21275,1280,43745,40227,3518


In [23]:
# Inspect the first rows before cleaning the Census workbook.
# These rows contain multi-level headers, not necessarily missing data.

display(preview.head(12))
display(preview.isna().sum().sort_values(ascending=False))

# Check rows that are completely empty
empty_rows = preview.isna().all(axis=1)
print("Completely empty rows:", empty_rows.sum())

# Check rows that contain at least one non-null value
print("Rows containing data:", (~empty_rows).sum())

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35
0,NaN,NaN,NaN,NaN,NaN,B-28 'OTHER WORKERS' BY DISTANCE FROM RESIDENCE TO PLACE OF WORK AND MODE OF TRAVEL TO PLACE OF WORK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Table,State,Distt.,Total/,Area Name,Mode of travel,Total,NaN,NaN,Distance to place of work in kilometres,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Name,Code,Code,Rural/,NaN,NaN,NaN,NaN,NaN,No travel,NaN,NaN,0-1,NaN,NaN,2-5,NaN,NaN,6-10,NaN,NaN,11-20,NaN,NaN,21-30,NaN,NaN,31-50,NaN,NaN,51+,NaN,NaN,Distance not stated,NaN,NaN
3,NaN,NaN,NaN,Urban/,NaN,NaN,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female
4,NaN,NaN,NaN,NaN,NaN,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,B0128,29,000,Total,KARNATAKA,All Modes,13222758,9794146,3428612,3865970,2518738,1347232,2080088,1461582,618506,2835872,2153578,682294,2036041,1609562,426479,1093090,914562,178528,497796,428421,69375,351308,304056,47252,337201,296020,41181,125392,107627,17765
7,B0128,29,000,Total,KARNATAKA,On foot,3145484,2068687,1076797,0,0,0,1550028,1008955,541073,1054869,684502,370367,540587,375230,165357,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,B0128,29,000,Total,KARNATAKA,Bicycle,748108,702745,45363,0,0,0,99227,92276,6951,360665,341230,19435,160661,151837,8824,64734,60152,4582,62821,57250,5571,0,0,0,0,0,0,0,0,0
9,B0128,29,000,Total,KARNATAKA,Moped/Scooter/Motor Cycle,1701363,1561606,139757,0,0,0,224185,206012,18173,618608,560571,58037,402054,368663,33391,258237,240114,18123,78503,74350,4153,53476,50394,3082,22555,21275,1280,43745,40227,3518


4     5
0     4
2     4
1     4
35    4
7     4
10    4
8     4
11    4
17    4
14    4
13    4
19    4
16    4
31    4
26    4
25    4
23    4
22    4
20    4
29    4
34    4
32    4
28    4
3     3
5     3
6     3
15    3
27    3
18    3
12    3
21    3
30    3
24    3
33    3
9     2
dtype: int64

Completely empty rows: 1
Rows containing data: 11


In [25]:
# Safe structural cleaning: do not impute Census values.
df_clean = preview.dropna(axis=0, how="all").copy()
df_clean = df_clean.dropna(axis=1, how="all")

print("Original shape:", preview.shape)
print("Cleaned shape:", df_clean.shape)

display(df_clean.head(12))

Original shape: (12, 36)
Cleaned shape: (11, 36)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35
0,NaN,NaN,NaN,NaN,NaN,B-28 'OTHER WORKERS' BY DISTANCE FROM RESIDENCE TO PLACE OF WORK AND MODE OF TRAVEL TO PLACE OF WORK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Table,State,Distt.,Total/,Area Name,Mode of travel,Total,NaN,NaN,Distance to place of work in kilometres,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Name,Code,Code,Rural/,NaN,NaN,NaN,NaN,NaN,No travel,NaN,NaN,0-1,NaN,NaN,2-5,NaN,NaN,6-10,NaN,NaN,11-20,NaN,NaN,21-30,NaN,NaN,31-50,NaN,NaN,51+,NaN,NaN,Distance not stated,NaN,NaN
3,NaN,NaN,NaN,Urban/,NaN,NaN,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female
4,NaN,NaN,NaN,NaN,NaN,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
6,B0128,29,000,Total,KARNATAKA,All Modes,13222758,9794146,3428612,3865970,2518738,1347232,2080088,1461582,618506,2835872,2153578,682294,2036041,1609562,426479,1093090,914562,178528,497796,428421,69375,351308,304056,47252,337201,296020,41181,125392,107627,17765
7,B0128,29,000,Total,KARNATAKA,On foot,3145484,2068687,1076797,0,0,0,1550028,1008955,541073,1054869,684502,370367,540587,375230,165357,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,B0128,29,000,Total,KARNATAKA,Bicycle,748108,702745,45363,0,0,0,99227,92276,6951,360665,341230,19435,160661,151837,8824,64734,60152,4582,62821,57250,5571,0,0,0,0,0,0,0,0,0
9,B0128,29,000,Total,KARNATAKA,Moped/Scooter/Motor Cycle,1701363,1561606,139757,0,0,0,224185,206012,18173,618608,560571,58037,402054,368663,33391,258237,240114,18123,78503,74350,4153,53476,50394,3082,22555,21275,1280,43745,40227,3518
10,B0128,29,000,Total,KARNATAKA,Car/Jeep/Van,534964,431023,103941,0,0,0,42278,33540,8738,123604,94791,28813,122953,96843,26110,120014,96892,23122,52702,43891,8811,35562,31075,4487,23895,22058,1837,13956,11933,2023


In [26]:
# Identify missingness; do not fill values yet.

missing_summary = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percentage": (
        df_clean.isna().mean() * 100
    ).round(2),
    "data_type": df_clean.dtypes.astype(str)
}).sort_values("missing_percentage", ascending=False)

display(missing_summary)

,missing_count,missing_percentage,data_type
4,4,36.36,str
0,3,27.27,str
2,3,27.27,str
1,3,27.27,str
35,3,27.27,object
7,3,27.27,object
10,3,27.27,object
8,3,27.27,object
11,3,27.27,object
17,3,27.27,object


In [27]:
# Diagnose the Census worksheet structure

print("Shape:", df_clean.shape)

print("\nColumn names:")
for i, col in enumerate(df_clean.columns):
    print(i, repr(col))

print("\nFirst 15 rows:")
display(df_clean.head(15))

print("\nNon-null values per column:")
display(df_clean.notna().sum().to_frame("non_null_count"))

print("\nData types:")
display(df_clean.dtypes.to_frame("dtype"))

Shape: (11, 36)

Column names:
0 0
1 1
2 2
3 3
4 4
5 5
6 6
7 7
8 8
9 9
10 10
11 11
12 12
13 13
14 14
15 15
16 16
17 17
18 18
19 19
20 20
21 21
22 22
23 23
24 24
25 25
26 26
27 27
28 28
29 29
30 30
31 31
32 32
33 33
34 34
35 35

First 15 rows:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35
0,NaN,NaN,NaN,NaN,NaN,B-28 'OTHER WORKERS' BY DISTANCE FROM RESIDENCE TO PLACE OF WORK AND MODE OF TRAVEL TO PLACE OF WORK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Table,State,Distt.,Total/,Area Name,Mode of travel,Total,NaN,NaN,Distance to place of work in kilometres,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Name,Code,Code,Rural/,NaN,NaN,NaN,NaN,NaN,No travel,NaN,NaN,0-1,NaN,NaN,2-5,NaN,NaN,6-10,NaN,NaN,11-20,NaN,NaN,21-30,NaN,NaN,31-50,NaN,NaN,51+,NaN,NaN,Distance not stated,NaN,NaN
3,NaN,NaN,NaN,Urban/,NaN,NaN,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female,Persons,Male,Female
4,NaN,NaN,NaN,NaN,NaN,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
6,B0128,29,000,Total,KARNATAKA,All Modes,13222758,9794146,3428612,3865970,2518738,1347232,2080088,1461582,618506,2835872,2153578,682294,2036041,1609562,426479,1093090,914562,178528,497796,428421,69375,351308,304056,47252,337201,296020,41181,125392,107627,17765
7,B0128,29,000,Total,KARNATAKA,On foot,3145484,2068687,1076797,0,0,0,1550028,1008955,541073,1054869,684502,370367,540587,375230,165357,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,B0128,29,000,Total,KARNATAKA,Bicycle,748108,702745,45363,0,0,0,99227,92276,6951,360665,341230,19435,160661,151837,8824,64734,60152,4582,62821,57250,5571,0,0,0,0,0,0,0,0,0
9,B0128,29,000,Total,KARNATAKA,Moped/Scooter/Motor Cycle,1701363,1561606,139757,0,0,0,224185,206012,18173,618608,560571,58037,402054,368663,33391,258237,240114,18123,78503,74350,4153,53476,50394,3082,22555,21275,1280,43745,40227,3518
10,B0128,29,000,Total,KARNATAKA,Car/Jeep/Van,534964,431023,103941,0,0,0,42278,33540,8738,123604,94791,28813,122953,96843,26110,120014,96892,23122,52702,43891,8811,35562,31075,4487,23895,22058,1837,13956,11933,2023



Non-null values per column:


,non_null_count
0,8
1,8
2,8
3,9
4,7
5,9
6,9
7,8
8,8
9,10



Data types:


,dtype
0,str
1,str
2,str
3,str
4,str
5,object
6,object
7,object
8,object
9,object


In [29]:
# Identify the Census header row safely, including NaN cells
keywords = [
    "table", "state", "distt.", "area name",
    "mode of travel", "total", "persons", "male", "female"
]

for i in range(min(15, len(df_clean))):
    row = df_clean.iloc[i].fillna("").astype(str).str.lower().tolist()

    matches = [
        keyword
        for keyword in keywords
        if any(keyword in value for value in row)
    ]

    print(f"Row {i}: {matches}")
    print(row[:10])
    print("-" * 80)

Row 0: ['mode of travel']
['', '', '', '', '', "b-28  'other workers'  by distance from residence to place of work and mode of travel to place of work", '', '', '', '']
--------------------------------------------------------------------------------
Row 1: ['table', 'state', 'distt.', 'area name', 'mode of travel', 'total']
['table', 'state', 'distt.', 'total/', 'area name', 'mode of travel', 'total ', '', '', 'distance to place of work in kilometres']
--------------------------------------------------------------------------------
Row 2: ['state']
['name', 'code', 'code', 'rural/', '', '', '', '', '', 'no travel']
--------------------------------------------------------------------------------
Row 3: ['persons', 'male', 'female']
['', '', '', 'urban/', '', '', 'persons', 'male', 'female', 'persons']
--------------------------------------------------------------------------------
Row 4: []
['', '', '', '', '', '1', '2', '3', '4', '5']
---------------------------------------------------

In [30]:
# Clean Census B-28 structure and reconstruct unique column names

# Use row 1 as the primary header
header = df_clean.iloc[1].fillna("").astype(str).str.strip()

# Rows 2–4 contain secondary header information
subheaders = df_clean.iloc[2:5].fillna("").astype(str)

columns = []
for j in range(df_clean.shape[1]):
    parts = [header.iloc[j]]

    for i in range(len(subheaders)):
        value = subheaders.iloc[i, j].strip()
        if value and value.lower() != "nan" and value not in parts:
            parts.append(value)

    name = "_".join(parts).strip("_")
    columns.append(name if name else f"column_{j}")

# Data begins at row 5
df_census = df_clean.iloc[5:].copy()
df_census.columns = columns
df_census = df_census.reset_index(drop=True)

# Remove completely empty rows and columns
df_census = df_census.dropna(axis=0, how="all")
df_census = df_census.dropna(axis=1, how="all")

# Convert numeric-looking values to numeric without imputing missing values
for col in df_census.columns:
    converted = pd.to_numeric(df_census[col], errors="coerce")

    # Convert only when every non-missing value is numeric
    non_missing = df_census[col].notna()
    if non_missing.any() and converted[non_missing].notna().all():
        df_census[col] = converted

print("Cleaned shape:", df_census.shape)
print("\nColumn names:")
print(df_census.columns.tolist())

display(df_census.head(10))

Cleaned shape: (6, 36)

Column names:
['Table_Name', 'State_Code', 'Distt._Code', 'Total/_Rural/_Urban/', 'Area Name', 'Mode of travel_1', 'Total_Persons_2', 'Male_3', 'Female_4', 'Distance to place of work in kilometres_No travel_Persons_5', 'Male_6', 'Female_7', '0-1_Persons_8', 'Male_9', 'Female_10', '2-5_Persons_11', 'Male_12', 'Female_13', '6-10_Persons_14', 'Male_15', 'Female_16', '11-20_Persons_17', 'Male_18', 'Female_19', '21-30_Persons_20', 'Male_21', 'Female_22', '31-50_Persons_23', 'Male_24', 'Female_25', '51+_Persons_26', 'Male_27', 'Female_28', 'Distance not stated_Persons_29', 'Male_30', 'Female_31']


,Table_Name,State_Code,Distt._Code,Total/_Rural/_Urban/,Area Name,Mode of travel_1,Total_Persons_2,Male_3,Female_4,Distance to place of work in kilometres_No travel_Persons_5,Male_6,Female_7,0-1_Persons_8,Male_9,Female_10,2-5_Persons_11,Male_12,Female_13,6-10_Persons_14,Male_15,Female_16,11-20_Persons_17,Male_18,Female_19,21-30_Persons_20,Male_21,Female_22,31-50_Persons_23,Male_24,Female_25,51+_Persons_26,Male_27,Female_28,Distance not stated_Persons_29,Male_30,Female_31
0,B0128,29,0,Total,KARNATAKA,All Modes,13222758,9794146,3428612,3865970,2518738,1347232,2080088,1461582,618506,2835872,2153578,682294,2036041,1609562,426479,1093090,914562,178528,497796,428421,69375,351308,304056,47252,337201,296020,41181,125392,107627,17765
1,B0128,29,0,Total,KARNATAKA,On foot,3145484,2068687,1076797,0,0,0,1550028,1008955,541073,1054869,684502,370367,540587,375230,165357,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,B0128,29,0,Total,KARNATAKA,Bicycle,748108,702745,45363,0,0,0,99227,92276,6951,360665,341230,19435,160661,151837,8824,64734,60152,4582,62821,57250,5571,0,0,0,0,0,0,0,0,0
3,B0128,29,0,Total,KARNATAKA,Moped/Scooter/Motor Cycle,1701363,1561606,139757,0,0,0,224185,206012,18173,618608,560571,58037,402054,368663,33391,258237,240114,18123,78503,74350,4153,53476,50394,3082,22555,21275,1280,43745,40227,3518
4,B0128,29,0,Total,KARNATAKA,Car/Jeep/Van,534964,431023,103941,0,0,0,42278,33540,8738,123604,94791,28813,122953,96843,26110,120014,96892,23122,52702,43891,8811,35562,31075,4487,23895,22058,1837,13956,11933,2023
5,B0128,29,0,Total,KARNATAKA,Tempo/Autorickshaw/Taxi,443851,359567,84284,0,0,0,33662,26842,6820,133414,96797,36617,109539,88127,21412,69538,59191,10347,33841,30067,3774,27364,25068,2296,19443,18367,1076,17050,15108,1942


In [31]:
# Validate Census records before proceeding

# Check geographic scope
print("Geographic coverage:")
display(
    df_census[["State_Code", "Distt._Code",
               "Total/_Rural/_Urban/", "Area Name"]]
    .drop_duplicates()
)

# Check travel modes and total worker counts
mode_col = "Mode of travel_1"
total_col = "Total_Persons_2"

display(
    df_census[[mode_col, total_col]]
    .assign(**{
        total_col: pd.to_numeric(df_census[total_col], errors="coerce")
    })
)

# Check whether the All Modes row equals the sum of individual modes
all_modes = df_census[
    df_census[mode_col].astype(str).str.strip().str.lower() == "all modes"
]

other_modes = df_census[
    df_census[mode_col].astype(str).str.strip().str.lower() != "all modes"
]

if not all_modes.empty:
    reported_total = pd.to_numeric(
        all_modes[total_col], errors="coerce"
    ).iloc[0]

    mode_sum = pd.to_numeric(
        other_modes[total_col], errors="coerce"
    ).sum()

    print("Reported All Modes total:", reported_total)
    print("Sum of listed mode rows:", mode_sum)
    print("Difference:", reported_total - mode_sum)

Geographic coverage:


,State_Code,Distt._Code,Total/_Rural/_Urban/,Area Name
0,29,0,Total,KARNATAKA


,Mode of travel_1,Total_Persons_2
0,All Modes,13222758
1,On foot,3145484
2,Bicycle,748108
3,Moped/Scooter/Motor Cycle,1701363
4,Car/Jeep/Van,534964
5,Tempo/Autorickshaw/Taxi,443851


Reported All Modes total: 13222758
Sum of listed mode rows: 6573770
Difference: 6648988


In [32]:
# Inspect all original rows and identify the travel-mode section

print("Original worksheet shape:", df_clean.shape)
print("Cleaned Census shape:", df_census.shape)

# Display all rows with the travel-mode column from the original data
display(df_clean.iloc[:, :8].to_string(index=True))

# Check whether the source contains more than six mode rows
print("\nRows in original worksheet:", len(df_clean))
print("\nRows in cleaned dataset:", len(df_census))

Original worksheet shape: (11, 36)
Cleaned Census shape: (6, 36)


"        0      1       2       3          4                                                                                                       5         6        7\n0     NaN    NaN     NaN     NaN        NaN  B-28  'OTHER WORKERS'  BY DISTANCE FROM RESIDENCE TO PLACE OF WORK AND MODE OF TRAVEL TO PLACE OF WORK       NaN      NaN\n1   Table  State  Distt.  Total/  Area Name                                                                                          Mode of travel    Total       NaN\n2    Name   Code    Code  Rural/        NaN                                                                                                     NaN       NaN      NaN\n3     NaN    NaN     NaN  Urban/        NaN                                                                                                     NaN   Persons     Male\n4     NaN    NaN     NaN     NaN        NaN                                                                                                       1         2  


Rows in original worksheet: 11

Rows in cleaned dataset: 6


In [33]:
# Rebuild Census dataset from the original worksheet structure

# Data starts at index 5 in the original worksheet
df_census = df_clean.loc[5:].copy()

# Keep rows with an actual travel-mode label in column 5
df_census = df_census[
    df_census.iloc[:, 5].notna()
].copy()

# Use the previously constructed column names
df_census.columns = columns

# Reset the index
df_census.reset_index(drop=True, inplace=True)

# Convert count columns to numeric, preserving missing values
for col in df_census.columns[6:]:
    df_census[col] = pd.to_numeric(
        df_census[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

print("Rebuilt shape:", df_census.shape)
display(df_census[[
    "Area Name",
    "Mode of travel_1",
    "Total_Persons_2"
]])

# Recheck total vs available mode rows
all_modes = df_census[
    df_census["Mode of travel_1"].astype(str).str.strip().str.lower()
    == "all modes"
]

other_modes = df_census[
    df_census["Mode of travel_1"].astype(str).str.strip().str.lower()
    != "all modes"
]

if not all_modes.empty:
    print("All Modes:", all_modes["Total_Persons_2"].iloc[0])
    print("Sum of listed modes:", other_modes["Total_Persons_2"].sum())

Rebuilt shape: (6, 36)


,Area Name,Mode of travel_1,Total_Persons_2
0,KARNATAKA,All Modes,13222758
1,KARNATAKA,On foot,3145484
2,KARNATAKA,Bicycle,748108
3,KARNATAKA,Moped/Scooter/Motor Cycle,1701363
4,KARNATAKA,Car/Jeep/Van,534964
5,KARNATAKA,Tempo/Autorickshaw/Taxi,443851


All Modes: 13222758
Sum of listed modes: 6573770


In [34]:
# Inspect all sheets in the original Census workbook

xls = pd.ExcelFile(CENSUS_FILE)

print("Available sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    temp = pd.read_excel(
        CENSUS_FILE,
        sheet_name=sheet,
        header=None
    )

    print(f"\nSheet: {sheet}")
    print("Shape:", temp.shape)

    # Display rows containing likely travel-mode labels
    if temp.shape[1] > 5:
        display(
            temp.loc[
                temp.iloc[:, 5].notna(),
                temp.columns[:8]
            ].tail(30)
        )

Available sheets: ['B-28']

Sheet: B-28
Shape: (1032, 36)


,0,1,2,3,4,5,6,7
999,B0128,29,584,Total,Ramanagara,Moped/Scooter/Motor Cycle,13196,12594
1000,B0128,29,584,Total,Ramanagara,Car/Jeep/Van,2385,2086
1001,B0128,29,584,Total,Ramanagara,Tempo/Autorickshaw/Taxi,6617,5261
1002,B0128,29,584,Total,Ramanagara,Bus,50464,39703
1003,B0128,29,584,Total,Ramanagara,Train,4013,3374
1004,B0128,29,584,Total,Ramanagara,Water transport,75,61
1005,B0128,29,584,Total,Ramanagara,Any other,997,901
1006,B0128,29,584,Total,Ramanagara,No travel,72825,45943
1007,B0128,29,584,Rural,Ramanagara,All Modes,112022,79105
1008,B0128,29,584,Rural,Ramanagara,On foot,17254,10915


In [35]:
# Reload the COMPLETE Census B-28 worksheet
raw_census = pd.read_excel(
    CENSUS_FILE,
    sheet_name="B-28",
    header=None
)

print("Full worksheet shape:", raw_census.shape)

# Reconstruct column names from the multi-row headers
header = raw_census.iloc[1].fillna("").astype(str).str.strip()
subheaders = raw_census.iloc[2:5].fillna("").astype(str)

columns = []
for j in range(raw_census.shape[1]):
    parts = [header.iloc[j]]
    for i in range(len(subheaders)):
        value = subheaders.iloc[i, j].strip()
        if value and value.lower() != "nan" and value not in parts:
            parts.append(value)

    name = "_".join(parts).strip("_")
    columns.append(name if name else f"column_{j}")

# Data starts at row 5; retain all geographic and travel-mode records
df_census = raw_census.iloc[5:].copy()
df_census.columns = columns
df_census = df_census.dropna(
    axis=0, how="all"
).reset_index(drop=True)

# Convert numeric columns while preserving true missing values
for col in df_census.columns[6:]:
    df_census[col] = pd.to_numeric(
        df_census[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

print("Full cleaned shape:", df_census.shape)

# Inspect Karnataka district names and codes
districts = (
    df_census[["Distt._Code", "Area Name"]]
    .drop_duplicates()
    .sort_values("Distt._Code")
)
display(districts)

Full worksheet shape: (1032, 36)
Full cleaned shape: (1025, 36)


,Distt._Code,Area Name
0,000,KARNATAKA
33,555,Belgaum
66,556,Bagalkot
99,557,Bijapur
132,558,Bidar
165,559,Raichur
198,560,Koppal
231,561,Gadag
264,562,Dharwad
297,563,Uttara Kannada


## 6. Load downloaded tabular files and profile schemas

This generic loader supports CSV, Excel, JSON and Parquet. PDF files are listed but not automatically converted to data because table extraction requires manual quality review. The code records row counts and column names; it does not guess the meaning of columns.

In [7]:
def load_tabular(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        # Try common encodings; preserve the first successful parse.
        last_error = None
        for enc in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
            try:
                return pd.read_csv(path, encoding=enc, low_memory=False)
            except Exception as e:
                last_error = e
        raise last_error
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported tabular format: {suffix}")

raw_tables = {}
schema_rows = []
for path in sorted(RAW.glob("*")):
    if path.suffix.lower() not in {".csv",".xlsx",".xls",".json",".parquet"}:
        continue
    try:
        df = load_tabular(path)
        raw_tables[path.name] = df
        schema_rows.append({
            "file":path.name, "rows":len(df), "columns":len(df.columns),
            "columns_list":" | ".join(map(str, df.columns)),
            "missing_cells":int(df.isna().sum().sum()),
            "duplicate_rows":int(df.duplicated().sum())
        })
        print(f"{path.name}: {df.shape}")
    except Exception as e:
        print("Could not parse", path.name, type(e).__name__, e)

schema_profile = pd.DataFrame(schema_rows)
display(schema_profile)
schema_profile.to_csv(PROCESSED / "schema_profile.csv", index=False)

""


## 7. Evidence classification before analysis

Create a manual mapping for each file. This step is important: a vehicle-registration stock series must not be treated as vehicles travelling on a particular Tuesday. Similarly, BMTC/BMRCL ridership is a separate public-transport indicator and must not be added to private road vehicles.

Use these recommended `measure_type` values:
- `observed_traffic_count`: observed vehicles passing a location during a stated interval.
- `observed_person_count`: observed people or passenger trips during a stated interval.
- `survey_estimate`: estimate expanded from a survey or origin-destination model.
- `mode_share`: proportions by travel mode, not raw counts.
- `registered_vehicle_stock`: vehicles registered as of a period.
- `public_transport_ridership`: passenger boardings/entries.
- `operations_metric`: fleet size, service km, utilisation, etc.
- `proxy_or_modelled`: modeled / inferred values.
- `unknown`: not yet validated; exclude from quantitative conclusions.

In [8]:
# Fill this after inspecting schema_profile and source metadata.
# Keep "unknown" until the meaning and provenance are confirmed.
file_metadata = pd.DataFrame([
    # Example structure (not data):
    # {"file":"my_traffic.csv", "source_url":"...", "publisher":"...",
    #  "measure_type":"observed_traffic_count", "evidence_grade":"B",
    #  "geography":"...", "period_start":"YYYY-MM-DD", "period_end":"YYYY-MM-DD",
    #  "time_column":"timestamp", "value_column":"vehicle_count",
    #  "mode_column":"vehicle_type", "public_transport":False,
    #  "methodology_notes":"..."}
], columns=[
    "file","source_url","publisher","measure_type","evidence_grade","geography",
    "period_start","period_end","time_column","value_column","mode_column",
    "public_transport","methodology_notes"
])
file_metadata.to_csv(PROCESSED / "file_metadata.csv", index=False)
display(file_metadata)

,file,source_url,publisher,measure_type,evidence_grade,geography,period_start,period_end,time_column,value_column,mode_column,public_transport,methodology_notes


## 8. Standardize actual timestamped traffic observations

The next function only prepares a dataset after you explicitly identify its timestamp and count columns. It does not manufacture dates, interpolate missing days, or convert registrations into traffic counts.

A row can represent a location/hour, a vehicle class/hour, or a daily total. Document that granularity before grouping.

In [9]:
def standardize_traffic(df, timestamp_col, count_col, location_col=None,
                        vehicle_type_col=None, public_transport_col=None):
    required = [timestamp_col, count_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    out = pd.DataFrame()
    out["timestamp"] = pd.to_datetime(df[timestamp_col], errors="coerce", dayfirst=False)
    out["vehicle_count"] = pd.to_numeric(
        df[count_col].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
    out["location"] = df[location_col].astype(str) if location_col and location_col in df else "unspecified"
    out["vehicle_type"] = df[vehicle_type_col].astype(str) if vehicle_type_col and vehicle_type_col in df else "unspecified"
    if public_transport_col and public_transport_col in df:
        out["public_transport"] = df[public_transport_col].astype(str).str.lower().isin(
            ["true","1","yes","public transport","bus","metro","train"]
        )
    else:
        out["public_transport"] = False

    out = out.dropna(subset=["timestamp","vehicle_count"])
    out = out[out["vehicle_count"] >= 0].copy()
    out["date"] = out["timestamp"].dt.date
    out["day_of_week"] = out["timestamp"].dt.day_name()
    out["weekday_type"] = np.where(out["timestamp"].dt.dayofweek < 5, "Weekday",
                                   np.where(out["timestamp"].dt.dayofweek == 5, "Saturday", "Sunday"))
    out["hour"] = out["timestamp"].dt.hour
    return out

# Example once a validated file has been selected:
# raw_df = raw_tables["validated_traffic_file.csv"]
# traffic = standardize_traffic(raw_df, timestamp_col="observed_at",
#                               count_col="vehicle_count",
#                               location_col="junction",
#                               vehicle_type_col="vehicle_class")

## 9. Weekday / weekend analysis for genuine traffic counts

This analysis runs only on a standardized dataset with true observation timestamps. If data contains multiple locations, the default is to report both (a) per-location averages and (b) sums only when locations are non-overlapping screenlines or a defensible citywide counting design. Summing overlapping sensors or multiple observations of the same vehicles can double-count traffic.

The daily mean is calculated over observed days only. The number of observed days is shown alongside the mean so incomplete coverage is visible.

In [10]:
def summarize_day_patterns(traffic, count_is_interval_volume=True):
    required = {"timestamp","vehicle_count","location","vehicle_type","public_transport","date","day_of_week","weekday_type"}
    missing = required - set(traffic.columns)
    if missing:
        raise ValueError(f"Traffic data is not standardized; missing: {missing}")

    # Main analysis excludes public transport records if they have been explicitly classified.
    road = traffic.loc[~traffic["public_transport"]].copy()
    public = traffic.loc[traffic["public_transport"]].copy()

    daily = (road.groupby(["date","location"], as_index=False)
             .agg(road_vehicle_count=("vehicle_count","sum"),
                  records=("vehicle_count","size")))
    daily["date"] = pd.to_datetime(daily["date"])
    daily["day_of_week"] = daily["date"].dt.day_name()
    daily["weekday_type"] = np.where(daily["date"].dt.dayofweek < 5, "Weekday",
                                     np.where(daily["date"].dt.dayofweek == 5, "Saturday", "Sunday"))

    # Per-location summaries; safer than citywide summing when sensor coverage is unknown.
    day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    by_day = (daily.groupby("day_of_week")
              .agg(mean_daily_count=("road_vehicle_count","mean"),
                   median_daily_count=("road_vehicle_count","median"),
                   observed_location_days=("road_vehicle_count","count"))
              .reindex(day_order).reset_index())
    by_daytype = (daily.groupby("weekday_type")
                  .agg(mean_location_day_count=("road_vehicle_count","mean"),
                       median_location_day_count=("road_vehicle_count","median"),
                       observed_location_days=("road_vehicle_count","count"))
                  .reset_index())
    return road, public, daily, by_day, by_daytype

# Example:
# road, public, daily_counts, weekday_summary, weekend_summary = summarize_day_patterns(traffic)
# display(weekday_summary, weekend_summary)

In [11]:
def plot_day_patterns(weekday_summary, weekend_summary):
    if not weekday_summary.empty:
        plt.figure(figsize=(10,4))
        plt.bar(weekday_summary["day_of_week"], weekday_summary["mean_daily_count"])
        plt.xticks(rotation=35, ha="right")
        plt.ylabel("Mean count per observed location-day")
        plt.title("Observed road-vehicle counts by day of week")
        plt.tight_layout()
        plt.show()

    if not weekend_summary.empty:
        plt.figure(figsize=(7,4))
        plt.bar(weekend_summary["weekday_type"], weekend_summary["mean_location_day_count"])
        plt.ylabel("Mean count per observed location-day")
        plt.title("Observed road-vehicle counts: weekday vs Saturday vs Sunday")
        plt.tight_layout()
        plt.show()

# plot_day_patterns(weekday_summary, weekend_summary)

## 10. Vehicle-type analysis

Do not assume a source's `vehicle_type` field means the same thing across datasets. Map raw labels into common groups only after checking the source's codebook. Keep buses as a separate class and decide whether the question is *all road vehicles* or *non-public-transport road vehicles*. For this project's primary total, exclude public transport passengers from passenger totals and report public-transport vehicles/operations separately rather than silently discarding buses from a road vehicle count.

Recommended categories:
- `two_wheeler`
- `car_jeep_van`
- `auto_taxi`
- `school_or_institutional_bus`
- `public_bus`
- `goods_vehicle`
- `other_or_unknown`

A public bus is physically on the road, so it may be included in an **all-road-vehicles** count but must be separated from private/non-public-transport vehicles. Do not add passenger ridership to vehicle counts.

In [12]:
VEHICLE_LABEL_MAP = {
    # Add normalized mappings only after reviewing original labels.
    "motorcycle":"two_wheeler",
    "motorbike":"two_wheeler",
    "scooter":"two_wheeler",
    "moped":"two_wheeler",
    "car":"car_jeep_van",
    "jeep":"car_jeep_van",
    "van":"car_jeep_van",
    "auto rickshaw":"auto_taxi",
    "autorickshaw":"auto_taxi",
    "taxi":"auto_taxi",
    "school bus":"school_or_institutional_bus",
    "institutional bus":"school_or_institutional_bus",
    "city bus":"public_bus",
    "public bus":"public_bus",
    "truck":"goods_vehicle",
    "lorry":"goods_vehicle",
}

def normalize_vehicle_type(value):
    label = str(value).strip().lower()
    return VEHICLE_LABEL_MAP.get(label, "other_or_unknown")

# Example:
# traffic["vehicle_group"] = traffic["vehicle_type"].map(normalize_vehicle_type)
# vehicle_mix = (traffic[~traffic["public_transport"]]
#                .groupby("vehicle_group")["vehicle_count"].sum()
#                .sort_values(ascending=False).rename("count").reset_index())
# display(vehicle_mix)

## 11. Public transport context — separate from the main road-commuter total

Maintain a separate table for BMTC, BMRCL, suburban rail and other public transport. Depending on the source, the measure may be passenger boardings, entries, fleet size, trips operated, route-km, or service frequency. Never add boardings to road vehicle counts. If public buses are present in traffic count data, keep them in the all-road-vehicles view and optionally exclude them for a private/non-public-transport view.

Suggested context fields:
`date_or_period`, `agency`, `measure`, `value`, `unit`, `geography`, `source_url`, `source_grade`, `notes`.

In [13]:
public_transport_context = pd.DataFrame(columns=[
    "date_or_period","agency","measure","value","unit","geography",
    "source_url","source_grade","notes"
])
public_transport_context.to_csv(PROCESSED / "public_transport_context.csv", index=False)
display(public_transport_context)

,date_or_period,agency,measure,value,unit,geography,source_url,source_grade,notes


## 12. Worker and student commuter estimates — prevent false precision

Census worker mode shares are not enough to estimate current daily student trips or weekday counts. For office, school and college commuting, use separate population denominators and clearly named assumptions.

A defensible estimate needs, at minimum:
1. population/enrolment or employment count and year;
2. share that travels on a given day;
3. share of each transport mode;
4. average occupancy for vehicle conversion;
5. school/college/office calendar, holidays and remote-work effects;
6. a defined city boundary and whether commuters entering from outside the city are counted.

If any of these are missing, publish a range or leave the estimate blank. Do not present a modeled estimate as an observed traffic count.

In [14]:
# Empty schema for future estimates. No values are fabricated in this notebook.
commuter_estimates = pd.DataFrame(columns=[
    "population_group", # office_workers / school_students / college_students
    "reference_year",
    "geography",
    "population_or_enrolment",
    "share_travelling_on_day",
    "mode",
    "mode_share",
    "average_occupancy",
    "estimated_people_trips_per_day",
    "estimated_vehicles_per_day",
    "estimate_low",
    "estimate_high",
    "source_url",
    "assumptions",
    "evidence_type" # should be survey_estimate, not observed count
])
commuter_estimates.to_csv(PROCESSED / "commuter_estimates_template.csv", index=False)

## 13. Quality checks and audit trail

Before any chart or claim is published:
- confirm raw file publisher, source URL, retrieval date, licence, data period and geographic boundary;
- inspect the data dictionary and units;
- check duplicates, missing timestamps, impossible negative values, repeated time/location keys and timezone;
- distinguish zero traffic from missing data;
- check whether counts are per 5/15/60 minutes, daily, or cumulative;
- check whether different sensors cover overlapping traffic;
- check holiday, school calendar, examination, strike, weather, and special-event effects where possible;
- document any manual mapping and retain the original raw labels;
- include `n observed days` / `n locations` with every daily mean.

In [15]:
def quality_report(df, timestamp_col=None, count_col=None, key_cols=None):
    report = {
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
    }
    if timestamp_col and timestamp_col in df:
        ts = pd.to_datetime(df[timestamp_col], errors="coerce")
        report["invalid_timestamps"] = int(ts.isna().sum())
        report["min_timestamp"] = str(ts.min())
        report["max_timestamp"] = str(ts.max())
    if count_col and count_col in df:
        values = pd.to_numeric(df[count_col], errors="coerce")
        report["invalid_numeric_counts"] = int(values.isna().sum())
        report["negative_counts"] = int((values < 0).sum())
    if key_cols and all(c in df for c in key_cols):
        report["duplicate_keys"] = int(df.duplicated(subset=key_cols).sum())
    return pd.Series(report, name="quality_report")

# Example after selecting a source:
# quality_report(raw_df, timestamp_col="timestamp", count_col="count",
#                key_cols=["timestamp","location","vehicle_type"])

## 14. Export analysis outputs

Use CSV outputs for reproducibility and sharing. Keep raw files untouched; transformed files should have clear names and the run date. Only export day-of-week summaries after a real, validated timestamped traffic dataset has been loaded.

In [16]:
# Save the source register and catalogue discovery output even if no traffic file is yet usable.
source_register.to_csv(PROCESSED / "source_register.csv", index=False)

# Example after validated analysis:
# road.to_csv(PROCESSED / f"road_traffic_observations_{RUN_DATE}.csv", index=False)
# daily_counts.to_csv(REPORTS / f"daily_road_vehicle_counts_{RUN_DATE}.csv", index=False)
# weekday_summary.to_csv(REPORTS / f"weekday_road_vehicle_summary_{RUN_DATE}.csv", index=False)
# weekend_summary.to_csv(REPORTS / f"weekday_weekend_comparison_{RUN_DATE}.csv", index=False)

print("Notebook workspace initialized:", ROOT.resolve())
print("No synthetic commuter/traffic observations have been created.")

Notebook workspace initialized: /home/dhirajpatra/Desktop/python/jupyter_notebooks/data_analysis/bengaluru_commuting_phase1
No synthetic commuter/traffic observations have been created.


## 15. Phase 1 deliverable checklist

- [x] Research questions, definitions and scope established.
- [x] Initial source register created with evidence grades and limitations.
- [x] OpenCity catalogue discovery and download workflow included.
- [x] Census commuting source documented; official workbook can be inspected.
- [x] Public transport context separated from the primary road-commuter analysis.
- [x] Day-of-week/weekend analysis implemented for validated, timestamped traffic counts.
- [x] Data quality checks and export templates included.
- [ ] Populate allowlist and download the specific source files after metadata review.
- [ ] Confirm which sources contain observed dated vehicle counts, rather than registrations or modeled values.
- [ ] Obtain/validate daily traffic counts across enough locations and dates for citywide comparisons.
- [ ] Obtain separate school/college enrolment and employment denominators before producing commuter estimates.

**Interpretation boundary:** This is a data-acquisition and analysis framework, not a claim that a complete citywide daily count dataset already exists publicly. Until validated dated observations are loaded, the weekday/weekend charts remain intentionally unexecuted.